In [ ]:
import yaml
import numpy as np
import polars as pl
from tqdm import tqdm
import statsmodels.api as sm
from scipy.stats import spearmanr

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Configuration and paths
mac = 20

# Maximum number of top-ranked variants per annotation (None = no limit)
max_num_variants = None

# Initialize new annotation variable as None
new_anno_local = None

# Load annotation configuration
config_path = "PATH_TO_FILE"

with open(config_path) as f:
    config = yaml.safe_load(f)

eur_samples_path = 'PATH_TO_FILE'

records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = pl.DataFrame(records).with_columns(
    pl.col("annotation_dir").cast(pl.Int8)
)

all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

In [ ]:
RAP_ANNO_DIR = "project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"
LOCAL_ANNO_DIR = "PATH_TO_FILE"

# ANNO_FILE = "annotations_fillna_ukbgym_with_mane.parquet"
ANNO_FILE = "annotations_with_all.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_ANNO_DIR}/{ANNO_FILE}
anno = pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")

existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]

regions_of_interest = ['ENSG00000130164']

anno = (
    anno
    .filter(
        # Filter to gene regions of interest
        (pl.col('region').is_in(regions_of_interest)),

        # Only SNPs
        # (pl.col('ref').str.len_chars()==1) & (pl.col('alt').str.len_chars()==1)
    )
    .select(
        set(['id', 'region']).union(set(existing_annos))
    )
    .collect(engine='streaming')
    .unique()
)

anno

# Pheno scatter plot

In [ ]:
# Get gene trait associations
RAP_DIR = 'project-REDACTED:/processed_data/REGENIE_results'

# ASSOC_FILE = 'loftee_mac20_associations_bh_corrected.parquet'
# ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR.parquet'
ASSOC_FILE = 'regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet'

LOCAL_DIR = 'PATH_TO_FILE'

!dx download {RAP_DIR}/{ASSOC_FILE} -o {LOCAL_DIR}

gene_trait_df = (
    pl.read_parquet(f'{LOCAL_DIR}/{ASSOC_FILE}')
    .filter(pl.col('pval_fdr')<=0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)

# CORR_FILE = 'regenie_127phenotypes_mac20_lofteeHC_EUR_correlations.parquet'
CORR_FILE = "regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per_correlations.parquet"
!dx download {RAP_DIR}/{CORR_FILE} -o {LOCAL_DIR}

loftee_corrs = (
    pl.read_parquet(f'{LOCAL_DIR}/{CORR_FILE}')
    .with_columns(
        loftee_corr = pl.col('correlation'),
        loftee_corr_abs = pl.col('correlation').abs(),
        loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
    )
    .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
)

gene_trait_df = (
    gene_trait_df
    .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
    .drop_nans()
    .filter(
        pl.col('region')=='ENSG00000130164',
        pl.col('phenotype')=='ldl_direct_int',
    )
    # .sort('loftee_corr_abs', descending=True)
    # .unique(subset=["region"], keep="first", maintain_order=True)
)

gene_trait_df

In [ ]:
RAP_APPV_DIR = "project-REDACTED:/processed_data/ukbgym/avg_pheno_per_var"
LOCAL_APPV_DIR = "PATH_TO_FILE"

# APPV_FILE = "loftee_mac20_quant_pheno_assocs_EURunrelated_appv_percentiles.parquet"
# APPV_FILE = "quant_pheno_loftee_mac20_EURunrelated_miss20per_appv_percentiles_small.parquet"
APPV_FILE = "quant_pheno_INT_loftee_mac20_EURunrelated_miss20per_appv_small.parquet"

!dx download {RAP_APPV_DIR}/{APPV_FILE} -o {LOCAL_APPV_DIR}/{APPV_FILE}
pheno_appv = pl.scan_parquet(f"{LOCAL_APPV_DIR}/{APPV_FILE}")

# Filter to variants in annotation set and low MAC
anno_keys = anno.select(pl.col('id').unique()).lazy()

# Merge phenotype data and annotation data
pheno_appv = (
    pheno_appv
    .join(
        anno_keys, on='id', how='semi'
    )
    .join(
        gene_trait_df.select(pl.col('phenotype').unique()).lazy(), on='phenotype', how='semi'
    )
    .filter(
        (pl.col('n_individuals') <= mac)
    )
    .select(
        ['id', 'phenotype', 'mean_pheno_value', 'n_individuals']
    )
    .collect(engine='streaming')
)

pheno_appv

In [ ]:
pheno_plot_df = (
    anno
    .join(
        pheno_appv, on='id', how='inner'
    )
)

pheno_plot_df

In [ ]:
# Select annotation to plot
selected_annotation = 'am_pathogenicity'

# Look up color and label from config
_anno_row = anno_config_df.filter(pl.col('annotation') == selected_annotation)
anno_color = _anno_row['color'][0]
anno_label = _anno_row['label'][0]

print(f"Annotation: {selected_annotation} → {anno_label} (color: {anno_color})")

In [ ]:
pheno_plot_df.filter(pl.col(selected_annotation) != 0)

In [ ]:
scatter_df = (
    pheno_plot_df
    .select(['id', selected_annotation, 'mean_pheno_value'])
    .drop_nulls()
    .with_columns(
        is_scored = pl.when(pl.col(selected_annotation) != 0)
            .then(pl.lit('Scored'))
            .otherwise(pl.lit('Not scored'))
    )
)

# Spearman correlation
rho, pval = spearmanr(
    scatter_df[selected_annotation].to_numpy(),
    scatter_df['mean_pheno_value'].to_numpy()
)
print(f"Spearman ρ = {rho:.3f} (p = {pval:.2e})")

(
    ggplot(scatter_df, aes(y='mean_pheno_value', x=selected_annotation))
    + geom_point(aes(color='is_scored'), alpha=0.5, size=1)
    + scale_color_manual(
        values={'Scored': anno_color, 'Not scored': 'black'},
    )
    + geom_smooth(method='glm', color='black', se=True)
    + labs(
        x=anno_label,
        y='Mean Phenotype Value',
        title=f'Spearman ρ = {rho:.3f}',
    )
    + theme_minimal()
    + theme(
        figure_size=(4, 4),
        title=element_text(size=13, lineheight=1.4),
        axis_text=element_text(size=13),
        axis_title=element_text(size=13, lineheight=1.4),
        plot_background=element_rect(fill="white", color="white"),
        legend_position='none',
    )
)

In [ ]:
pheno_plot_df.filter(pl.col('loftee_hc') != 0)

In [ ]:
# LOFTEE HC box plot
loftee_anno = 'loftee_hc'
_loftee_row = anno_config_df.filter(pl.col('annotation') == loftee_anno)
loftee_color = _loftee_row['color'][0]
loftee_label = _loftee_row['label'][0]

box_df = (
    pheno_plot_df
    .select(['id', loftee_anno, 'mean_pheno_value'])
    .drop_nulls()
    .with_columns(
        group = pl.when(pl.col(loftee_anno) == 1).then(pl.lit('True')).otherwise(pl.lit('False'))
    )
)

# Spearman correlation
rho, pval = spearmanr(
    box_df[loftee_anno].to_numpy(),
    box_df['mean_pheno_value'].to_numpy()
)
print(f"Spearman ρ = {rho:.3f} (p = {pval:.2e})")

(
    ggplot(box_df, aes(x='group', y='mean_pheno_value'))
    + geom_boxplot(color='black', width=0.5)
    + geom_jitter(aes(color='group', alpha='group'), width=0.2, size=1)
    + scale_color_manual(values={'False': 'black', 'True': loftee_color})
    + scale_alpha_manual(values={'False': 0, 'True': 0.5})
    + labs(
        x=loftee_label,
        y='Mean Phenotype Value',
        title=f'Spearman ρ = {rho:.3f}',
    )
    + theme_minimal()
    + theme(
        figure_size=(3, 4),
        legend_position='none',
        title=element_text(size=13, lineheight=1.4),
        axis_text=element_text(size=13),
        axis_title=element_text(size=13, lineheight=1.4),
        plot_background=element_rect(fill="white", color="white"),
    )
)

# Pheno scatter plot

In [ ]:
RAP_APPV_DIR = "project-REDACTED:/processed_data/ukbgym/avg_pheno_per_var"
LOCAL_APPV_DIR = "PATH_TO_FILE"

APPV_FILE = "quant_pheno_INT_loftee_mac20_EURunrelated_miss20per_appv_small.parquet"

!dx download {RAP_APPV_DIR}/{APPV_FILE} -o {LOCAL_APPV_DIR}/{APPV_FILE}
pheno_appv = pl.scan_parquet(f"{LOCAL_APPV_DIR}/{APPV_FILE}")

# Filter to variants in annotation set and low MAC
anno_keys = anno.select(pl.col('id').unique()).lazy()

pheno_appv = (
    pheno_appv
    .join(
        anno_keys, on='id', how='semi'
    )
    .filter(
        (pl.col('n_individuals') <= mac)
    )
    .select(
        ['id', 'phenotype', 'mean_pheno_value', 'n_individuals']
    )
    .collect(engine='streaming')
)

pheno_appv